In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss


# ============================================================
# LOAD V6 DATA
# ============================================================

DATA_PATH = "../data/processed/features_v6.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")


# ============================================================
# CLEAN TARGET
# ============================================================

df = df.dropna(subset=["FTR"]).copy()

df["MatchDateTime"] = pd.to_datetime(df["MatchDateTime"])
df = df.sort_values("MatchDateTime").reset_index(drop=True)

print(f"\nRows after target cleaning: {len(df)}")

print("\nTarget distribution:")
print(df["FTR"].value_counts())

print("\nGames by season:")
print(df["Season"].value_counts().sort_index())


# ============================================================
# TARGET
# ============================================================

target = "FTR"


# ============================================================
# CURRENT BEST 5-FEATURE MODEL
# ============================================================

base_features = [
    "EloDiff",
    "AwayElo",
    "XGDDiffLast5",
    "AwayGDPerGame",
    "AwayPointsLast5"
]


# ============================================================
# CANDIDATE FEATURES
#
# These are the same candidate pool you've been using,
# excluding features already in the current model.
# ============================================================

candidate_features = [
    "ShotOTDiffLast5",
    "GoalAgainstDiffLast5",
    "GDPerGameDiff",
    "PPGDiff",
    "GoalsPerGameDiff",
    "GoalsAgainstPerGameDiff",
    "ShotDiffLast5",
    "GoalDiffLast5",
    "HomeElo",
    "HomePPG",
    "AwayPPG",
    "HomeGDPerGame",
    "HomeGoalsPerGame",
    "AwayGoalsAgainstPerGame",
    "HomePointsLast5",
    "HomeGoalsLast5",
    "AwayGoalsLast5",
    "HomeGoalsAgainstLast5",
    "AwayGoalsAgainstLast5",
    "HomeShotsAgainstLast5",
    "AwayShotsAgainstLast5",
    "HomeShotsOnTargetLast5",
    "AwayShotsOnTargetLast5",
    "HomeShotsOnTargetAgainstLast5",
    "AwayShotsOnTargetAgainstLast5",
    "XGForDiffPg",
    "XGAgainstDiffPg",
    "XGDDiff",
    "XGForDiffLast5",
    "XGAgainstDiffLast5"
]

candidate_features = [
    f for f in candidate_features
    if f not in base_features
]


# ============================================================
# REMOVE FEATURES WE ALREADY KNOW ARE BAD
#
# From the previous V6 analysis:
# ============================================================

bad_features = [
    # Add any features you've definitively ruled out here
]

candidate_features = [
    f for f in candidate_features
    if f not in bad_features
]


print("\nBase features:")
for f in base_features:
    print("-", f)

print(f"\nCandidate features: {len(candidate_features)}")


# ============================================================
# MODEL
#
# KEEPING THE EXISTING LOGISTIC SETUP.
#
# IMPORTANT:
# This is NOT multinomial logistic regression.
# ============================================================

def make_model():

    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )


# ============================================================
# SEASON EVALUATION
# ============================================================

def evaluate_features(features):

    results = []

    seasons = sorted(df["Season"].unique())

    for season in seasons:

        # Do not evaluate first season because there is
        # no previous season training data in this setup.
        if season == seasons[0]:
            continue

        train = df[df["Season"] < season].copy()
        test = df[df["Season"] == season].copy()

        train = train.dropna(subset=features)
        test = test.dropna(subset=features)

        X_train = train[features]
        y_train = train[target]

        X_test = test[features]
        y_test = test[target]

        model = make_model()

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)

        classes = model.classes_

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        logloss = log_loss(
            y_test,
            probabilities,
            labels=classes
        )

        # Multiclass Brier score
        y_test_encoded = pd.get_dummies(
            y_test
        ).reindex(
            columns=classes,
            fill_value=0
        ).values

        brier = np.mean(
            np.sum(
                (probabilities - y_test_encoded) ** 2,
                axis=1
            )
        )

        results.append({
            "Season": season,
            "Games": len(test),
            "Accuracy": accuracy,
            "LogLoss": logloss,
            "Brier": brier
        })

    results_df = pd.DataFrame(results)

    return results_df


# ============================================================
# BASELINE
# ============================================================

print("\n" + "=" * 70)
print("CURRENT BEST 5-FEATURE MODEL")
print("=" * 70)

baseline_results = evaluate_features(base_features)

print(
    "\nFeatures:",
    " + ".join(base_features)
)

print(
    f"\nMean Accuracy: "
    f"{baseline_results['Accuracy'].mean():.6f}"
)

print(
    f"Mean Log Loss: "
    f"{baseline_results['LogLoss'].mean():.6f}"
)

print(
    f"Mean Brier: "
    f"{baseline_results['Brier'].mean():.6f}"
)

print("\nSeason results:")
print(
    baseline_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


baseline_accuracy = baseline_results["Accuracy"].mean()
baseline_logloss = baseline_results["LogLoss"].mean()
baseline_brier = baseline_results["Brier"].mean()


# ============================================================
# SEARCH FOR 6TH FEATURE
# ============================================================

print("\n" + "=" * 70)
print("SEARCHING FOR 6TH FEATURE")
print("=" * 70)

search_results = []

for i, feature in enumerate(candidate_features, 1):

    features = base_features + [feature]

    print(
        f"[{i}/{len(candidate_features)}] {feature}"
    )

    try:

        results = evaluate_features(features)

        search_results.append({
            "Features": " + ".join(features),
            "AddedFeature": feature,
            "NumFeatures": len(features),
            "MeanAccuracy": results["Accuracy"].mean(),
            "MeanLogLoss": results["LogLoss"].mean(),
            "MeanBrier": results["Brier"].mean()
        })

    except Exception as e:

        print(f"ERROR with {feature}: {e}")


# ============================================================
# RESULTS
# ============================================================

search_df = pd.DataFrame(search_results)

search_df = search_df.sort_values(
    ["MeanLogLoss", "MeanBrier"],
    ascending=[True, True]
).reset_index(drop=True)


print("\n" + "=" * 70)
print("TOP 20 SIX-FEATURE COMBINATIONS")
print("=" * 70)

print(
    search_df.head(20).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# COMPARE TO BASELINE
# ============================================================

search_df["AccuracyChange"] = (
    search_df["MeanAccuracy"] - baseline_accuracy
)

search_df["LogLossChange"] = (
    search_df["MeanLogLoss"] - baseline_logloss
)

search_df["BrierChange"] = (
    search_df["MeanBrier"] - baseline_brier
)


print("\n" + "=" * 70)
print("TOP 20 WITH CHANGES FROM BASELINE")
print("=" * 70)

print(
    search_df[
        [
            "Features",
            "MeanAccuracy",
            "MeanLogLoss",
            "MeanBrier",
            "AccuracyChange",
            "LogLossChange",
            "BrierChange"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

OUTPUT_PATH = "../data/processed/sext_feature_search_v6.csv"

search_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"\nSaved: {OUTPUT_PATH}"
)


# ============================================================
# BEST FEATURE SET
# ============================================================

best = search_df.iloc[0]

print("\n" + "=" * 70)
print("BEST SIX-FEATURE SET")
print("=" * 70)

print(
    f"\nFeatures: {best['Features']}"
)

print(
    f"Mean Accuracy: {best['MeanAccuracy']:.6f}"
)

print(
    f"Mean Log Loss: {best['MeanLogLoss']:.6f}"
)

print(
    f"Mean Brier: {best['MeanBrier']:.6f}"
)

print(
    f"\nAccuracy change: {best['AccuracyChange']:+.6f}"
)

print(
    f"Log Loss change: {best['LogLossChange']:+.6f}"
)

print(
    f"Brier change: {best['BrierChange']:+.6f}"
)